# Otimizador de Rotas Hospitalares

## Chamadas da biblioteca route_engine

As funções importadas na próxima célula formam a API pública usada pelo notebook, pelo servidor MCP e pelo Route Assistant. Abaixo está o papel de cada chamada.

| Chamada | O que faz | Retorno |
| --- | --- | --- |
| listar_unidades() | Carrega as unidades cadastradas e converte seus Plus Codes em coordenadas. | Lista de dicionários com nome, tipo, prioridade, demanda e coords. |
| exibir_mapa_unidades() | Cria o mapa geral da sede e de todas as unidades. | Objeto folium.Map. |
| calcular_melhor_rota_entrega(...) | Calcula a melhor ordem de entrega usando ruas reais e algoritmo genético. | Dicionário com rota, viagens, distâncias, restrições e metadados. |
| criar_mapa_rota(resultado) | Desenha o resultado já calculado em mapa interativo e animado. | Objeto folium.Map. |

### Como chamar o cálculo

O único argumento obrigatório é nomes_locais: uma lista com nomes completos ou trechos não ambíguos das unidades. A rota sempre começa na sede. Os parâmetros opcionais permitem simular outro veículo ou operação:

- capacidade_kg: carga máxima do veículo;
- autonomia_km: limite de distância de cada viagem, já incluindo o retorno à sede;
- prioridades: dicionário no formato {nome_da_unidade: nivel}, em que 1 é alta, 2 é média e 3 é normal;
- demandas_kg: dicionário no formato {nome_da_unidade: peso_em_kg};
- configuracao_ga: ajustes técnicos opcionais do algoritmo, como tamanho_populacao, geracoes, taxa_mutacao, elitismo e semente.

A função retorna rota, viagens, distancia_total_km, restricoes, comparativo_referencia e algoritmo. Ela lança ValueError para lista vazia, unidade inexistente/ambígua, duplicidade ou restrições inviáveis, como uma demanda maior que a capacidade.

## Roteiro da demonstração

Este notebook apresenta a solução para distribuição de medicamentos e insumos entre unidades de saúde. O motor combina malha viária do OpenStreetMap, algoritmo genético e restrições operacionais para definir rotas viáveis.

**Roteiro:** conhecer os pontos, calcular uma rota padrão, aplicar restrições e interpretar o resultado no mapa.

## O problema resolvido

A operação parte do **Hub Logístico Saúde Centro** e atende unidades selecionadas. O algoritmo genético evita testar todas as permutações possíveis e procura uma boa ordem considerando:

- distância real pelas ruas;
- prioridade clínica das entregas;
- capacidade do veículo;
- autonomia máxima por viagem;
- retorno à sede quando uma nova saída é necessária.

In [1]:
from IPython.display import HTML, display

from route_engine import (
    CONFIGURACAO_ALGORITMO_GENETICO,
    SEDE,
    VEICULO_PADRAO,
    calcular_melhor_rota_entrega,
    criar_mapa_rota,
    exibir_mapa_unidades,
    listar_unidades,
)

print(f"Sede: {SEDE['nome']}")
print(f"Veículo padrão: {VEICULO_PADRAO['capacidade_kg']:.0f} kg | {VEICULO_PADRAO['autonomia_km']:.0f} km")
print(
    "Algoritmo genético: "
    f"{CONFIGURACAO_ALGORITMO_GENETICO['tamanho_populacao']} indivíduos | "
    f"{CONFIGURACAO_ALGORITMO_GENETICO['geracoes']} gerações"
)

Sede: Hub Logístico Saúde Centro
Veículo padrão: 120 kg | 45 km
Algoritmo genético: 120 indivíduos | 300 gerações


## Pontos cadastrados

A biblioteca converte os Plus Codes cadastrados em coordenadas e disponibiliza as unidades prontas para cálculo e visualização.

In [2]:
unidades = listar_unidades()

linhas = ''.join(
    f"<tr><td>{indice}</td><td>{unidade['nome']}</td>"
    f"<td>{unidade['tipo']}</td><td>{unidade['prioridade']}</td>"
    f"<td>{unidade['demanda_kg']:.0f} kg</td></tr>"
    for indice, unidade in enumerate(unidades, start=1)
)
display(HTML(
    "<table><thead><tr><th>#</th><th>Unidade</th><th>Tipo</th>"
    "<th>Prioridade</th><th>Demanda</th></tr></thead><tbody>"
    f"{linhas}</tbody></table>"
))
print(f"Total de unidades disponíveis: {len(unidades)}")

#,Unidade,Tipo,Prioridade,Demanda
1,Hospital Leforte Liberdade,Hospital Geral,2,35 kg
2,Hospital Alemão Oswaldo Cruz,Hospital Privado,1,30 kg
3,Hospital Pró Matre Paulista,Maternidade,1,25 kg
4,Hospital TotalCor,Hospital Cardiológico,3,20 kg
5,Hospital Samaritano Paulista,Hospital Geral,2,30 kg
6,Hospital IGESP Paulista,Hospital Geral,2,25 kg
7,HCor - Hospital do Coração,Hospital Cardiológico,1,35 kg
8,Hospital Luz Vila Mariana,Hospital Geral,2,25 kg
9,UBS Humaitá,UBS / Posto de Saúde,3,15 kg
10,Hospital Municipal Infantil Menino Jesus,Hospital Infantil,1,30 kg


Total de unidades disponíveis: 12


In [3]:
# Mapa geral: verde = sede, azul = UBS, vermelho = demais unidades.
exibir_mapa_unidades()

## Cenário 1 - Entrega padrão

O motor usa a capacidade e a autonomia padrão. A primeira execução pode levar mais tempo porque baixa a malha de ruas; as próximas reutilizam o cache local.

In [4]:
locais_cenario_padrao = [
    'Hospital IGESP Paulista',
    'HCor - Hospital do Coração',
    'Hospital Luz Vila Mariana',
]

resultado_padrao = calcular_melhor_rota_entrega(locais_cenario_padrao)

In [5]:
def exibir_resumo(resultado):
    comparativo = resultado['comparativo_referencia']
    print(f"Distância otimizada: {resultado['distancia_total_km']:.2f} km")
    print(f"Viagens necessárias: {resultado['total_viagens']}")
    print(f"Economia versus ordem informada: {comparativo['economia_distancia_km']:.2f} km ({comparativo['economia_percentual']:.1f}%)")
    print('\\nOrdem recomendada:')
    for parada in resultado['rota']:
        print(
            f"  {parada['ordem_na_viagem']}. {parada['nome']} | "
            f"viagem {parada['viagem']} | prioridade {parada['prioridade_rotulo']} | "
            f"{parada['demanda_kg']:.0f} kg"
        )

exibir_resumo(resultado_padrao)

Distância otimizada: 10.97 km
Viagens necessárias: 1
Economia versus ordem informada: 0.41 km (3.6%)
\nOrdem recomendada:
  1. HCor - Hospital do Coração | viagem 1 | prioridade alta | 35 kg
  2. Hospital Luz Vila Mariana | viagem 1 | prioridade media | 25 kg
  3. Hospital IGESP Paulista | viagem 1 | prioridade media | 25 kg


In [6]:
# Linhas de cores diferentes representam viagens distintas.
criar_mapa_rota(resultado_padrao)

## Cenário 2 - Restrições operacionais

Simulamos um veículo menor e com autonomia limitada. Prioridades e demandas abaixo substituem apenas os valores deste cenário; o cadastro original permanece inalterado.

In [7]:
locais_cenario_restrito = [
    'Hospital Alemão Oswaldo Cruz',
    'Hospital Pró Matre Paulista',
    'HCor - Hospital do Coração',
    'Hospital Luz Vila Mariana',
    'UBS Humaitá',
]

resultado_restrito = calcular_melhor_rota_entrega(
    locais_cenario_restrito,
    capacidade_kg=80,
    autonomia_km=25,
    prioridades={
        'Hospital Alemão Oswaldo Cruz': 1,
        'Hospital Pró Matre Paulista': 1,
        'HCor - Hospital do Coração': 1,
        'Hospital Luz Vila Mariana': 2,
        'UBS Humaitá': 3,
    },
    demandas_kg={
        'Hospital Alemão Oswaldo Cruz': 30,
        'Hospital Pró Matre Paulista': 25,
        'HCor - Hospital do Coração': 35,
        'Hospital Luz Vila Mariana': 25,
        'UBS Humaitá': 15,
    },
    configuracao_ga={'tamanho_populacao': 80, 'geracoes': 150, 'semente': 42},
)

exibir_resumo(resultado_restrito)
for viagem in resultado_restrito['viagens']:
    print(f"\nViagem {viagem['numero']}: {viagem['carga_kg']:.0f} kg | {viagem['distancia_km']:.2f} km")
    for parada in viagem['paradas']:
        print(f"  - {parada['nome']} ({parada['prioridade_rotulo']})")

Distância otimizada: 18.38 km
Viagens necessárias: 2
Economia versus ordem informada: 0.71 km (3.7%)
\nOrdem recomendada:
  1. HCor - Hospital do Coração | viagem 1 | prioridade alta | 35 kg
  2. Hospital Alemão Oswaldo Cruz | viagem 1 | prioridade alta | 30 kg
  1. Hospital Pró Matre Paulista | viagem 2 | prioridade alta | 25 kg
  2. Hospital Luz Vila Mariana | viagem 2 | prioridade media | 25 kg
  3. UBS Humaitá | viagem 2 | prioridade normal | 15 kg

Viagem 1: 65 kg | 7.52 km
  - HCor - Hospital do Coração (alta)
  - Hospital Alemão Oswaldo Cruz (alta)

Viagem 2: 65 kg | 10.86 km
  - Hospital Pró Matre Paulista (alta)
  - Hospital Luz Vila Mariana (media)
  - UBS Humaitá (normal)


In [8]:
criar_mapa_rota(resultado_restrito)

## Conclusão

O motor entrega uma sequência de atendimento, divide saídas quando necessário e apresenta um mapa interpretável. A mesma biblioteca é utilizada pelo servidor MCP e pelo Route Assistant, que transforma os resultados em instruções e relatórios operacionais.

Para detalhes de cada chamada pública, consulte route_engine/README.md.